# Text Mining Project - Final Model
**Spring Semester 2025/2026**

Group 11:
- Ana Macedo
- Carlota Pires
- Francisca Calçoa
- Francisca Martins



<hr>
<a class="anchor" id="one-bullet"> 
<d style="color:white;">

# 1. Imports and Load Data
</a> 
</d>   

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
train = pd.read_csv('../Datasets/train.csv')
test = pd.read_csv('../Datasets/test.csv')

X_train_raw = train['text'].astype(str)
y_train_raw = train['label']

X_test_raw = test['text'].astype(str)

<hr>
<a class="anchor" id="two-bullet"> 
<d style="color:white;">

# 2. Final Model Pipeline
</a> 
</d>   
!! here we must put the final model as a pipeline!!!

depois de ter o modelo ajustar isto

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=train['label'].nunique()
)

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(X_train_raw, y_train_raw)
test_dataset = TweetDataset(X_test_raw)

training_args = TrainingArguments(
    output_dir="./bertweet-final",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()



### Export Test Predictions

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=1)

submission = pd.DataFrame({
    "id": test["id"],
    "label": pred_labels
})

submission.to_csv("pred_xx.csv", index=False)